## Домашнее задание 7

В этом задании мы обучим русскоязычного ассистента для ответов на произвольные вопросы с помощью Parameter-Efficient Fine-tuning.

А точнее, вам предстоит реализовать два метода PEFT: _Low-Rank Adaptation (LoRA)_ и _Prompt Tuning_, а так же дообучить с их помощью 700M модель.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch

device = torch.device('cuda')

Для обучения мы будем использовать датасет [`lksy/ru_instruct_gpt4`](https://huggingface.co/datasets/lksy/ru_instruct_gpt4). Он содержит наборы инструкций и ответов к ним, сгенерированных с помощью GPT-4.

In [6]:
!pip install -q datasets --upgrade

In [8]:
BASE_FILE_PATH = '/content/drive/MyDrive/data/'

In [9]:
from datasets import load_dataset


data_files = {'train': BASE_FILE_PATH + 'ru_instruct_gpt4_train.tsv', 'test': BASE_FILE_PATH + 'ru_instruct_gpt4_test.tsv'}
dataset = load_dataset('csv', data_files=data_files, sep='\t')

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'full_output'],
        num_rows: 12931
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'full_output'],
        num_rows: 1125
    })
})

In [10]:
dataset['train'][0]

{'instruction': 'Посоветуй, какие три вида спорта сделают лето интереснее и забавнее для 12-летнего ребенка.',
 'input': None,
 'output': 'Летом 12-летнему ребенку могут быть интересны следующие виды спорта: катание на роликах, которое тренирует координацию и баланс; плавание, укрепляющее мышцы и обеспечивающее охлаждение в жару; и фрисби, улучшающее ловкость, скорость и активное общение на свежем воздухе.',
 'full_output': '1. Плавание: Летом отлично провести время на пляже или в бассейне, занимаясь плаванием. Это не только интересное и веселое развлечение, но еще и полезное для здоровья и физического развития ребенка.\n\n2. Велосипедный спорт: Прогулки на велосипеде разнообразят летние дни, позволят 12-летнему ребенку активно проводить время на свежем воздухе и укреплять мышцы ног.\n\n3. Бадминтон: Это веселая и энергичная игра развивает координацию, внимание и гибкость, и при этом не требует сложного оборудования. Бадминтон можно играть как в парке, так и на личном дворе.'}

In [ ]:
dataset['train'][2]

{'instruction': 'Напишите краткое содержание рассказа на основе указанных персонажей и событий.',
 'input': 'Персонажи: Иван, Ольга, кот Мурзик; события: отпуск, ссора, примирение',
 'output': 'Во время отпуска Иван и Ольга ссорятся из-за некоторых разногласий. В этот момент их кот Мурзик совершает поступок, который помогает супругам осознать свою ошибку и примириться друг с другом.',
 'full_output': None}

Для обучения возьмем трансформерную seq2seq модель от Сбера – ruT5. Прочитать про T5 можно [тут](https://huggingface.co/docs/transformers/model_doc/t5) или [тут](https://arxiv.org/pdf/1910.10683). Если кратко, то модель обучалась решать любую задачу как задачу генерации. Для этого на ей вход Encoder'а подавалась последовательность, в которой некоторые фразы закрывались специальными символами. Модель должна была предсказать, что скрыто.

<img src="https://i.ibb.co/pXkQPkF/T5-example.png" alt="drawing" width="500"/>

ruT5, как следует из названия, обучалась на русском языке, поэтому она отлично подойдет для наших данных.

In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("ai-forever/ruT5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("ai-forever/ruT5-large", device_map='cuda')
sum(p.numel() for p in model.parameters())

737668096

Для формирования входной последовательности мы объединим поля `instruction` и `input` в одну строку, выходом будет поле `output`. `full_output` – это более развернутый ответ на тот же запрос, который мы будем игнорировать, так как часто он пустой.

__Задание 1.__ Допишите функцию `preprocess`. Она принимает на вход словарь с полями `instruction`, `input` и `output`, а возвращает словарь с токенами входных текстов без паддингов (`input_ids`) и токены выходных текстов (`labels`). Ограничьте максимальную длину входной последовательности 100 токенами, а выходной – 200.

In [14]:
def preprocess(samples: dict) -> dict:
    input_texts = [f'{instr}\n\n{inp}' for instr, inp in zip(samples['instruction'], samples['input'])]

    # Токенизируем входные тексты с ограничением длины 100 токенов
    inputs = tokenizer(input_texts, truncation=True, max_length=100, padding=False)

    # Токенизируем выходные тексты с ограничением длины 200 токенов
    outputs = tokenizer(samples['output'], truncation=True, max_length=200, padding=False)

    # Создаем словарь результатов
    result = {
        "input_ids": inputs.input_ids,
        "labels": outputs.input_ids
    }

    return result

In [15]:
tokenized_dataset = dataset.map(preprocess, batched=True, remove_columns=['full_output'])

Map:   0%|          | 0/12931 [00:00<?, ? examples/s]

Map:   0%|          | 0/1125 [00:00<?, ? examples/s]

Для оценки качества предсказаний будем использовать BLEU.

In [17]:
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.5 MB/s eta 0:00:00


In [18]:
from evaluate import load

bleu_metric = load("bleu")

def compute_metrics(values):
    predictions, labels = values
    pred_text = tokenizer.batch_decode(np.argmax(predictions, axis=1))
    labels = tokenizer.batch_decode(labels)

    return bleu_metric.compute(predictions=pred_text, references=[[l] for l in labels])

Для начала проверим, как модель генерирует ответ без дообучения и удостоверимся, что совсем не так, как мы хотим.

In [19]:
lm_text='Придумайте название для книги на основе следующего описания сюжета.\n\nМолодой парень находит портал в параллельный мир, где его жизнь намного хуже, чем она была до этого.'

input_ids = torch.tensor([tokenizer.encode(lm_text)]).to(device)

outputs = model.generate(input_ids, eos_token_id=tokenizer.eos_token_id, early_stopping=True)

print(tokenizer.decode(outputs[0][1:]))

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:679: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


<extra_id_0> и назовите ее.</s>


## Low-Rank Adaptation (LoRA)

В этой секции мы самостоятельно напишем LoRA адаптер для эффективного дообучения. Напомним, как он выглядит. К квадратной матрице весов добавляется добавка в виде произведения матриц: $W' = W + AB$. При этом матрицы $A$ и $B$ имеют пониженный ранг. Заметьте, что, в отличие от метода Adapters, тут нет активации между слоями.

<img src="https://i.ibb.co/Sxq33GY/lora-1.png" alt="drawing" width="300"/>

__Задание 2.__ Допишите класс-обертку LoRALayer. Он модифицирует линейный слой с помощью LoRA. Несмотря на то, что в оригинальной реализации матрица A инициализируется нормальным распределением, на практике лучше себя показывает [Kaiming Uniform Initialization](https://pytorch.org/docs/stable/nn.init.html#torch.nn.init.kaiming_uniform_), используйте ее с параметром $a=\sqrt{5}$ (это стандартное значение). Матрицу $B$ инициализируйте нулями.

In [20]:
from torch import nn
import torch.nn.functional as F
import torch


class LoRALayer(nn.Module):
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()

        self.module = module
        in_features = module.in_features
        out_features = module.out_features

        # Создаем параметры для адаптеров
        self.adapter_A = nn.Parameter(torch.empty(in_features, rank))
        self.adapter_B = nn.Parameter(torch.zeros(rank, out_features))

        # Инициализация матрицы A с помощью kaiming_uniform_ с a=5**0.5
        nn.init.kaiming_uniform_(self.adapter_A, a=5**0.5)
        # Матрица B уже инициализирована нулями

    def forward(self, hidden_states):
        """
        Применяет линейный слой ко входу с учетом LoRA адаптера
        """
        # Применяем оригинальный линейный слой
        original_output = self.module(hidden_states)

        # Применяем LoRA адаптер (матрица A, затем матрица B)
        lora_output = hidden_states @ self.adapter_A @ self.adapter_B

        # Объединяем оригинальный выход с выходом адаптера
        return original_output + lora_output

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("ai-forever/ruT5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("ai-forever/ruT5-large", device_map='cuda')

__Задание 3.__ Напишите функцию `add_lora` для добавления LoRA адаптера ко всем слоям `q` и `v` механизмов внимания в нашей модели (Self-Attention и Cross-Attention). Список всех модулей можно получить из `model.modules()`. Ранг LoRA выставите равным 8. Функция возвращает модифицированную модель.

In [ ]:
def add_lora(model, lora_rank=8):
    device = next(iter(model.parameters())).device
    # ваш код здесь

In [ ]:
lora_model = add_lora(model, lora_rank=8)

Отключим градиенты у всех остальных параметров, чтобы училась только LoRA.

In [ ]:
for name, param in lora_model.named_parameters():
    if 'adapter_A' in name or 'adapter_B' in name:
        continue
    else:
        param.requires_grad = False

In [ ]:
model_params = sum(p.numel() for p in lora_model.parameters())
trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print(
    f'All params: %s | Trainable params: %s | Trainable %%: %.4f' % \
    (model_params, trainable_params, round(trainable_params / model_params, 4))
)

Для обучения добавим `DataCollator`, который будет добавлять паддинги к входам и выходам модели.

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=lora_model, padding=True, max_length=128, label_pad_token_id=-100
)

__Задание 4.__ Используйте `transformers.Seq2SeqTrainer` для того, чтобы дообучить LoRA на наших данных. Пока что достаточно обучить одну эпоху, чтобы проверить, что все работает. Выберите размер батча не меньше 4, больше скорость обучения и обучайте в пониженной точности `float16` (параметр `fp16=True` в `Seq2SeqTrainingArguments`). Если все хорошо, то после одной эпохи вы должны получить BLEU не меньше 0.03.

Для проверки сдайте предсказания вашей модели в виде `solution.tsv` файла с одной строковой колонкой "prediction", в которой будут записаны предсказания для соответствующих строк из выборки `ru_instruct_gpt4_grader.tsv`. Мы измерим BLEU для ваших предсказаний на нашей приватной выборке.

## Prompt Tuning

Теперь настало очередь реализации второго метода PEFT – Prompt Tuning. Напомним, что он добавляет обучаемый промпт в начало входного текста.

<img src="https://i.ibb.co/54bv1Gt/prompt-tuning-1.png" alt="drawing" width="450"/>


Так же, как и для LoRA, мы напишем обертку, которая будет реализовывать добавление обучаемых эмбеддингов.

В плане реализации жизнь немного облегчается тем, что у T5 нет эмбеддингов позиций. Информация о позициях в ней добавляется с помощью Relative Positional Embeddings. Подробно об этом мы поговорим не сейчас. Важно то, что обучаемые эмбеддинги просто конкатенируются с эмбеддингами слов и дальше обрабатываются как обычно. Правда, из-за этого размер маски внимания перестает соответствовать числу токенов, но это мы исправим с помощью DataCollator в следующем задании.

__Задание 5.__ Допишите класс `PromptTuningEmbedding`. Он принимает слой эмбеддингов и размер обучаемого промпта. В методе forward он добавляет обучаемые эмбеддинги к эмбеддингам токенов и возвращает результат. Этот слой будет использоваться вместо слоя эмбеддингов в Encoder'е модели.

In [ ]:
class PromptTuningEmbedding(nn.Module):
    """
    Используется вместо слоя эмбеддингов в модели. Добавляет обучаемые эмбеддинги перед эмбеддингами токенов.
    """

    def __init__(self, embed_tokens: nn.Embedding, prompt_size: int):
        super().__init__()
        self.embed_tokens = embed_tokens
        self.prompt_size = prompt_size
        self.learnable_prompts = nn.Parameter(...)

    def forward(self, input_ids):
        """
        input_ids: индексы токенов
        return: конкатенация обучаемых и необучаемых эмбеддингов
        """

        return embeds

Как мы сказали, добавление обучаемый ембеддингов увеличивает длину последовательности, что приводит к несоответствию между ее размером и размером маски внимания. Для того, чтобы это исправить, будем увеличивать размер маски до подачи в модель.

__Задание 6.__ Допишите класс `PTDataCollator`, наследуемый от `DataCollatorForSeq2Seq`. При его вызове он превращает последовательности в тензоры с помощью `DataCollatorForSeq2Seq`, а затем добавляет матрицу из единиц в начало маски внимания для Encoder'а, чтобы выровнить ее размер.

In [ ]:
from transformers import DataCollatorForSeq2Seq

class PTDataCollator(DataCollatorForSeq2Seq):
    def __init__(self, prompt_size: int, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.prompt_size = prompt_size  # число обучаемых эмбеддингов

    def __call__(self, *args, **kwargs) -> dict:
        """
        Вызывает DataCollatorForSeq2Seq для входа и
        дополняет полученный attention_mask матрицей из единиц
        """
        # ваш код здесь
        pass

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("ai-forever/ruT5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("ai-forever/ruT5-large", device_map='cuda')

In [ ]:
prompt_size = 20
lora_rank = 8

model.encoder.embed_tokens = PromptTuningEmbedding(model.encoder.embed_tokens, prompt_size).to(device)
pt_lora_model = add_lora(model, lora_rank=lora_rank)

In [ ]:
for name, param in pt_lora_model.named_parameters():
    if 'learnable_prompts' in name or 'adapter_A' in name or 'adapter_B' in name:
        continue
    else:
        param.requires_grad = False

In [ ]:
model_params = sum(p.numel() for p in pt_lora_model.parameters())
trainable_params = sum(p.numel() for p in pt_lora_model.parameters() if p.requires_grad)
print(
    f'All params: %s | Trainable params: %s | Trainable %%: %.4f' % \
    (model_params, trainable_params, round(trainable_params / model_params, 4))
)

In [ ]:
data_collator = PTDataCollator(
    prompt_size, tokenizer, model=pt_lora_model, padding=True, max_length=128, label_pad_token_id=-100
)

__Задание 7.__ Обучите полученную модель с LoRA и Prompt Tuning. Все тесты должна пройти модель, обученная за одну эпоху, но, если вы хотите получить ассистента, который ошибается в ответах ещё меньше, то лучше обучать модель около 4 эпох.

Для проверки сдайте предсказания вашей модели в виде `solution.tsv` файла с одной строковой колонкой "prediction", в которой будут записаны предсказания для соответствующих строк из выборки `ru_instruct_gpt4_grader.tsv`. Мы измерим BLEU для ваших предсказаний на нашей приватной выборке.

Поздравляем! Теперь у вас есть модель, которая может отвечать на произвольные вопросы (хоть часто и не очень верно), собранная с минимальными вычислительными затратами. Можете попробовать попросить у нее что-нибудь не очень сложное :)   
Кстати, похожим образом обучалась ChatGPT на одном из этапов.

Помимо генерации, PEFT можно (и нужно) применять к задаче классификации. При этом часто отставание от полного дообучения для классификации будет меньше, так как классификация проще сама по себе.